In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from elecc_colombia.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR

In [3]:
total_actas_downloaded_df = pd.read_csv(EXTERNAL_DATA_DIR / "actas_download_status_manually.csv")
total_actas_downloaded_df.head()

,DEPARTAMENTO,URL_ACTAS,DOWNLOAD STATUS,COMPUTER,100_PERC_PUBLISHED,NUMBER_MESAS,NOTE
0,AMAZONAS,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,176,NaN
1,ANTIOQUIA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,15801,NaN
2,ARAUCA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,662,NaN
3,ATLANTICO,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,6192,NaN
4,BOGOTA D.C.,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,17164,NaN


## Check Number of Actas Downloaded

In [14]:
vuelta = 2

actas_downloaded_df = pd.read_csv(INTERIM_DATA_DIR / f"vuelta{vuelta:02}"/ f"actas_mac_log.csv")

#concatenate the two dataframes
actas_downloaded_df = pd.concat([actas_downloaded_df, pd.read_csv(INTERIM_DATA_DIR / f"vuelta{vuelta:02}"/ f"actas_windows_log.csv")])

#order records by DEPARTAMENTO and by RETRIEVE_DATE
actas_downloaded_df = actas_downloaded_df.sort_values(by=["DEPARTAMENTO", "RETRIEVAL_TIMESTAMP"])
#eliminate duplocates based on DEPARTAMENTO, MUNICIPIO, ZONA, and MESA, keeping the last record (the most recent one)
actas_downloaded_df = actas_downloaded_df.drop_duplicates(subset=["DEPARTAMENTO", "MUNICIPIO", "ZONA", "PUESTO", "MESA"], keep="last")

actas_downloaded_df

,RETRIEVAL_TIMESTAMP,DEPARTAMENTO,MUNICIPIO,ZONA,PUESTO,MESA,ACTA_PDF
17852,2026-06-23 19:32:09,AMAZONAS,010 — EL ENCANTO (100%),Zona 00,00 - CORREGIMIENTO DEPARTAMENTAL,Mesa 1,data/raw/AMAZONAS/010_EL_ENCANTO_ZONA_00_00_CO...
17853,2026-06-23 19:32:10,AMAZONAS,010 — EL ENCANTO (100%),Zona 00,00 - CORREGIMIENTO DEPARTAMENTAL,Mesa 2,data/raw/AMAZONAS/010_EL_ENCANTO_ZONA_00_00_CO...
17854,2026-06-23 19:32:11,AMAZONAS,010 — EL ENCANTO (100%),Zona 00,00 - CORREGIMIENTO DEPARTAMENTAL,Mesa 3,data/raw/AMAZONAS/010_EL_ENCANTO_ZONA_00_00_CO...
17855,2026-06-23 19:32:24,AMAZONAS,013 — LA CHORRERA (100%),Zona 00,00 - CORREGIMIENTO DEPARTAMENTAL,Mesa 1,data/raw/AMAZONAS/013_LA_CHORRERA_ZONA_00_00_C...
17856,2026-06-23 19:32:25,AMAZONAS,013 — LA CHORRERA (100%),Zona 00,00 - CORREGIMIENTO DEPARTAMENTAL,Mesa 2,data/raw/AMAZONAS/013_LA_CHORRERA_ZONA_00_00_C...
...,...,...,...,...,...,...,...
19664,2026-06-23 20:33:46,VICHADA,002 — SANTA ROSALIA (100%),Zona 00,00 - PUESTO CABECERA MUNICIPAL,Mesa 8,data/raw/VICHADA/002_SANTA_ROSALIA_ZONA_00_00_...
19665,2026-06-23 20:33:47,VICHADA,002 — SANTA ROSALIA (100%),Zona 00,00 - PUESTO CABECERA MUNICIPAL,Mesa 9,data/raw/VICHADA/002_SANTA_ROSALIA_ZONA_00_00_...
19666,2026-06-23 20:33:55,VICHADA,002 — SANTA ROSALIA (100%),Zona 99,10 - GUACACIAS,Mesa 1,data/raw/VICHADA/002_SANTA_ROSALIA_ZONA_99_10_...
19667,2026-06-23 20:34:00,VICHADA,002 — SANTA ROSALIA (100%),Zona 99,12 - FLOR AMARILLO,Mesa 1,data/raw/VICHADA/002_SANTA_ROSALIA_ZONA_99_12_...


In [16]:
#Count number of records (this is number of mesas) by departamento and compare with total_actas_downloaded_df
actas_count_by_dep = actas_downloaded_df.groupby("DEPARTAMENTO").size()
total_actas_df = total_actas_downloaded_df.copy()
#in total_actas_downloaded_df, the column is called NUMBER_MESAS
# add a column to total_actas_downloaded_df with the count of actas downloaded by departamento
total_actas_df = total_actas_df.merge(actas_count_by_dep.rename("NUMBER_MESAS_DOWNLOADED"), left_on="DEPARTAMENTO", right_index=True)
total_actas_df["NUMBER_MESAS_DOWNLOADED"] = total_actas_df["NUMBER_MESAS_DOWNLOADED"].fillna(0).astype(int)
total_actas_df

,DEPARTAMENTO,URL_ACTAS,DOWNLOAD STATUS,COMPUTER,100_PERC_PUBLISHED,NUMBER_MESAS,NOTE,NUMBER_MESAS_DOWNLOADED
2,ARAUCA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,662,NaN,652
3,ATLANTICO,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,6192,NaN,6190
4,BOGOTA D.C.,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,17164,NaN,17164
5,BOLIVAR,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,5355,NaN,5320
7,CALDAS,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,2492,NaN,2492
8,CAQUETA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,983,NaN,923
9,CASANARE,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,1020,NaN,1020
10,CAUCA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,3427,NaN,3401
11,CESAR,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,2754,NaN,2754
12,CHOCO,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,1252,THERE ARE 1253,1252


In [ ]:
# To debug, get a subset of the records where the NUMBER_MESAS mismatch the NUMBER_MESAS_DOWNLOADED
total_actas_df[total_actas_df['NUMBER_MESAS'] != total_actas_df["NUMBER_MESAS_DOWNLOADED"]]

,DEPARTAMENTO,URL_ACTAS,DOWNLOAD STATUS,COMPUTER,100_PERC_PUBLISHED,NUMBER_MESAS,NOTE,NUMBER_MESAS_DOWNLOADED
2,ARAUCA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,662,NaN,652
3,ATLANTICO,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,6192,NaN,6190
5,BOLIVAR,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,5355,NaN,5320
8,CAQUETA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,983,NaN,923
10,CAUCA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,3427,NaN,3401
14,CORDOBA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,4180,NaN,4179
16,GUAINIA,https://e14segundavueltapresidente.registradur...,DOWNLOADED,MAC,YES,117,THERE ARE 115,115
23,NORTE DE SAN,https://e14segundavueltapresidente.registradur...,DOWNLOADED,WINDOWS-VM,YES,4079,NaN,4078


## Compare Number of Actas Reported in Website